In [1]:
import os
import concurrent.futures
import openai
import pandas as pd
import PyPDF2
from tqdm import tqdm
import time

# Config
FOLDER_PATH = "cot_papers_selected"
CSV_PATH = "cot_papers_selected_summary.csv"
MODEL_NAME = "gpt-4.1-2025-04-14"  # or your "gpt-4.1" model name depending on your access
MAX_WORKERS = 8  # adjust based on your system
MAX_RETRIES = 3

openai.api_key = os.environ["OPENAI_API_KEY"]

summarize_prompt = """
You are an expert research assistant.
Given a research paper, extract the following information in a strictly structured format. Be precise and avoid adding any extra commentary.
Output format:

method: <brief description of how the method reduces redundancy in token usage (1 paragraph)>
algorithm-pseudocode: <core pseudocode of the method>
datasets:

    dataset1: accuracy: <value>, tokenefficiency: <percentage>

    dataset2: accuracy: <value>, tokenefficiency: <percentage>
    ...(add as many datasets as applicable)

Extraction rules:

    If any information is missing in the paper, write: Not reported

    Always include units (%, etc.)

    Use consistent formatting for easy parsing.

    Focus only on token redundancy reduction aspects.
NOTE: Do not approximate any values only report what is present.
"""

def read_pdf_text(filepath, max_pages=20):
    text = ""
    try:
        with open(filepath, 'rb') as f:
            reader = PyPDF2.PdfReader(f)
            for page in reader.pages[:max_pages]:
                text += page.extract_text() or ""
    except Exception as e:
        print(f"Error reading {filepath}: {e}")
    return text

def call_openai_api(content):
    retries = 0
    while retries < MAX_RETRIES:
        try:
            response = openai.ChatCompletion.create(
                model=MODEL_NAME,
                messages=[
                    {"role": "system", "content": "You are a helpful research assistant."},
                    {"role": "user", "content": summarize_prompt + "\n\nPaper content:\n" + content}
                ],
                max_tokens=2048,
                temperature=0
            )
            return response['choices'][0]['message']['content']
        except Exception as e:
            retries += 1
            print(f"Retry {retries} due to error: {e}")
            time.sleep(2 ** retries)
    return "Error after retries"

def process_file(filepath):
    filename = os.path.basename(filepath)
    title = os.path.splitext(filename)[0]
    text = read_pdf_text(filepath)
    summary = call_openai_api(text)
    return {"title": title, "summary": summary}

def main():
    files = [os.path.join(FOLDER_PATH, f) for f in os.listdir(FOLDER_PATH) if f.endswith('.pdf')]
    results = []

    with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(process_file, file): file for file in files}
        for future in tqdm(concurrent.futures.as_completed(futures), total=len(futures), desc="Processing Papers"):
            try:
                result = future.result()
                results.append(result)
            except Exception as e:
                print(f"Failed processing file: {e}")

    df = pd.DataFrame(results)
    df.to_csv(CSV_PATH, index=False)
    print(f"Saved to {CSV_PATH}")

if __name__ == "__main__":
    main()

Processing Papers: 100%|███████████████████████████████████████████████| 179/179 [06:22<00:00,  2.14s/it]

Saved to cot_papers_selected_summary.csv
